In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler
from itertools import combinations

| Column | Description |
|--------|-------------|
| **Date** | The date of the data point |
| **Ticker** | The stock symbol |
| **Open** | The opening price of the stock on that day |
| **High** | The highest price reached on that day |
| **Low** | The lowest price reached on that day |
| **Close** | The closing price of the stock |
| **Adjusted** | The adjusted closing price after stock splits and dividends |
| **Returns** | Daily percentage return based on close prices |
| **Volume** | The volume of shares traded that day |

In [2]:
# ref: https://www.kaggle.com/datasets/nikitamanaenkov/stock-portfolio-data-with-prices-and-indices?resource=download
data = pd.read_csv('../datasets//Portfolio_prices.csv')

In [3]:
data.columns = data.columns.str.lower()

In [4]:
data

,date,ticker,open,high,low,close,adjusted,returns,volume
0,2020-01-03,JPM,137.500000,139.229996,137.080002,138.339996,119.874138,-0.013197,10386800
1,2020-01-03,KO,54.320000,54.990002,54.090000,54.689999,46.494698,-0.005455,11354500
2,2020-01-03,LMT,404.019989,417.170013,403.000000,413.739990,359.312317,0.035981,2990100
3,2020-01-03,MS,51.220001,51.450001,50.830002,51.200001,43.273079,-0.016142,6706000
4,2020-01-03,MSCI,262.200012,263.579987,259.269989,260.750000,248.019470,-0.019921,459700
...,...,...,...,...,...,...,...,...,...
35392,2025-03-21,ADAP,0.307000,0.310000,0.280000,0.291000,0.291000,0.039286,5008800
35393,2025-03-21,AAPL,211.559998,218.839996,211.279999,218.270004,218.270004,0.019477,93954500
35394,2025-03-21,SPY,559.280029,564.890015,558.030029,563.979980,563.979980,0.000330,83666800
35395,2025-03-21,HUM,266.140015,269.309998,263.230011,266.170013,266.170013,-0.002473,1963600


### Create sophisticated technical indicators and identify momentum shifts across stocks.


Step 1: Calculate Technical Indicators

For **each ticker individually**, calculate:

| Indicator | Formula / Window |
|-----------|------------------|
| **5-day SMA** | Simple Moving Average of `Close` price over 5 days |
| **20-day EMA** | Exponential Moving Average of `Close` price over 20 days |
| **RSI (14-day)** | Relative Strength Index over 14-day periods |
| **Bollinger Bands** | Upper and Lower bands (±2 standard deviations from 20-day SMA) |
| **VWAP (5-day)** | Volume-Weighted Average Price using a 5-day rolling window |


**Relative Strength Index (RSI) Formula**:

$$
\text{RSI} = 100 - \frac{100}{1 + \text{RS}}
$$

Where:

$$
\text{RS} = \frac{\text{Average Gain}}{\text{Average Loss}} \quad \text{(over 14 periods)}
$$


Average Gain / Loss (Smoothed):

$$
\text{Avg Gain} = \frac{(\text{Previous Avg Gain} \times 13) + \text{Current Gain}}{14}
$$

$$
\text{Avg Loss} = \frac{(\text{Previous Avg Loss} \times 13) + \text{Current Loss}}{14}
$$


Step-by-Step Calculation RSI:

| Step | Description |
|------|-------------|
| 1 | Calculate daily price change: `Change = Close_today - Close_yesterday` |
| 2 | Separate Gains and Losses: |
|    | - Gain = `max(Change, 0)` |
|    | - Loss = `max(-Change, 0)` |
| 3 | Calculate **Average Gain** and **Average Loss** over 14 periods (using the formula above) |
| 4 | Calculate **RS**: `Average Gain / Average Loss` |
| 5 | Calculate **RSI**: `100 - (100 / (1 + RS))` |

**Bollinger Bands** consist of three lines:

| Band | Formula |
|------|---------|
| **Middle Band** | 20-day Simple Moving Average (SMA) |
| **Upper Band** | Middle Band + (2 × Standard Deviation) |
| **Lower Band** | Middle Band - (2 × Standard Deviation) |


$$
\text{Middle Band} = \text{SMA}_{20}
$$

$$
\text{Upper Band} = \text{SMA}_{20} + (2 \times \sigma_{20})
$$

$$
\text{Lower Band} = \text{SMA}_{20} - (2 \times \sigma_{20})
$$

Where:
- $\text{SMA}_{20}$ = Simple Moving Average over 20 periods
- $\sigma_{20}$ = Standard Deviation over 20 periods

**vwap**:

| Component | Description |
|-----------|-------------|
| **Price × Volume** | Dollar value traded for each day |
| **Sum of (Price × Volume)** | Total dollar value over the window |
| **Sum of Volume** | Total shares traded over the window |
| **VWAP** | Total dollar value / Total shares |

Step 2: Create Signal Indicator

Generate trading signals based on the following logic:

| Signal | Condition |
|--------|-----------|
| **Buy** | `Close` crosses **above** 20-day EMA **AND** RSI < 30 |
| **Sell** | `Close` crosses **below** 20-day EMA **AND** RSI > 70 |
| **Hold** | Otherwise |


Step 3: Calculate Signal Success Rate

For each ticker, calculate the **success rate** of signals by checking if price moves **up/down 3%** within the next **5 trading days**.

In [5]:
# Data:     [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]

# Window size = 3:

# Step 1: [1, 2, 3] → mean = 2
# Step 2:    [2, 3, 4] → mean = 3
# Step 3:       [3, 4, 5] → mean = 4

| Parameter | Description | Example |
|-----------|-------------|---------|
| **`window`** | Size of the rolling window | `3` (3 rows) |
| **`min_periods`** | Minimum observations required | `1` (allow partial windows) |
| **`center`** | Center the window | `True` / `False` |
| **`closed`** | Which side is closed | `'right'`, `'left'`, `'both'` |

In [6]:
# step 1
# simple moving average
data['sma_5'] = data.groupby(['ticker'])['close'].rolling(window=5, min_periods=1).mean().reset_index(level=0, drop=True)

# exponential moving average
data['ema_20'] = data.groupby(['ticker'])['close'].ewm(span=20, adjust=False).mean().reset_index(level=0, drop=True)

# rsi
def rsi(data):
    # 1. calculate daily price change
    changes = data.diff()
    
    # 2. separate Gains and Losses (separate positive and negative values)
    gain = changes.where(changes>0, 0) # is equivalent to --> np.where(changes > 0, changes, 0)
    loss = changes.where(changes<0, 0)
    
    # 3. calculate average Gain and aerage Loss over 14 periods
    gain_avg = gain.rolling(window=14, min_periods=14).mean() # only calculate the average when there are at least 14 non-null values available.
    loss_avg = loss.rolling(window=14, min_periods=14).mean()
    
    # 4. calculate RS (Relative Strength)
    rs = gain_avg / loss_avg
    
    # 5. calculate rsi
    rsi = 100 - (100 / (1 + rs))    

    return rsi

data['rsi_14'] = data.groupby('ticker')['close'].transform(rsi)

# bollinger bands
def bollinger_band(data, band_type='upper'):
    middle_band = data.rolling(window=20, min_periods=1).mean() # sma 20 days
    std_20 = data.rolling(window=20, min_periods=1).std() # std 20 days
    upper_band = middle_band + 2 * std_20
    lower_band = middle_band - 2 * std_20

    if band_type=='upper':
        return upper_band
    elif band_type=='lower':
        return lower_band
    else:
        raise ValueError()
        
data['bollinger_upper_band'] = data.groupby('ticker')['close'].transform(bollinger_band, band_type='upper')  
data['bollinger_lower_band'] = data.groupby('ticker')['close'].transform(bollinger_band, band_type='lower')  

# vwap 
def vwap(data):
    value = data['close'] / data['volume']
    total_value = value.rolling(window=5, min_periods=5).sum()
    total_volume = data['volume'].rolling(window=5, min_periods=5).sum()

    return total_value / total_volume

data['vwap'] = data.groupby('ticker', group_keys=False).apply(vwap, include_groups=False)  

In [7]:
# step 2
close_prev = data.groupby('ticker')['close'].transform(lambda x: x.shift(1))
ema_prev = data.groupby('ticker')['ema_20'].transform(lambda x: x.shift(1))

buy_condition = (
    (data['close'] > data['ema_20']) & 
    (close_prev <= ema_prev) & # prev row of close <= prev row of ema_20
    (data['rsi_14'] < 30) # RSI < 30
)

sell_condition = (
    (data['close'] < data['ema_20']) & 
    (close_prev >= ema_prev) &
    (data['rsi_14'] > 70)
)

data['signal'] = np.select([buy_condition, sell_condition], ['buy', 'sell'], default='hold')

In [8]:
data[data['signal']!='hold'].head(20)

,date,ticker,open,high,low,close,adjusted,returns,volume,sma_5,ema_20,rsi_14,bollinger_upper_band,bollinger_lower_band,vwap,signal
364,2020-01-23,GRWG,4.590000,4.650000,4.200000,4.370000,4.370000,-0.070213,421600,4.568000,4.436566,830.768074,4.920209,3.972648,9.022484e-11,sell
371,2020-01-23,IBKR,48.779999,49.169998,48.180000,48.240002,46.726879,-0.019313,731100,49.640001,48.541413,435.294162,50.860406,46.549595,1.145634e-10,sell
407,2020-01-27,MS,53.000000,53.689999,52.860001,53.110001,44.887371,-0.026398,10649100,54.993999,53.552236,315.310989,57.798075,49.261924,3.933014e-13,sell
409,2020-01-27,MSFT,161.149994,163.380005,160.199997,162.279999,154.907867,-0.016723,32078100,165.247998,162.815453,382.769306,169.083358,156.772889,2.505785e-13,sell
425,2020-01-27,AAPL,77.514999,77.942497,76.220001,77.237503,74.798180,-0.029405,161940000,79.038002,77.496570,312.785997,81.488456,73.964045,5.239364e-15,sell
472,2020-01-29,TMUS,81.870003,81.889999,80.089996,80.260002,78.487083,-0.017144,2602400,81.280002,80.441529,588.092751,83.200555,77.679446,1.989511e-11,sell
499,2020-01-30,IBKR,47.470001,48.590000,47.470001,48.580002,47.056213,0.013139,305000,47.616001,48.200918,-14733.939471,50.598996,46.237848,2.070286e-10,buy
516,2020-01-31,MS,53.200001,53.279999,51.889999,52.259998,44.458851,-0.028985,11286900,53.378000,53.495193,12700.190738,57.359135,49.666864,5.648342e-13,sell
521,2020-01-31,PG,125.989998,126.949997,124.459999,124.620003,110.133362,-0.010560,6612900,125.470000,124.848122,735.382701,127.577165,121.976834,2.933196e-12,sell
523,2020-01-31,TMUS,80.589996,80.790001,79.160004,79.190002,77.440727,-0.021621,2863600,80.554002,80.365288,2752.597976,83.084691,77.720310,1.388822e-11,sell


In [9]:
# step 3 --> no idea what it says!

### Analyze inter-stock correlations and construct optimal portfolios.

Step 1: Create a return matrix (pivot table) with Dates as rows, Tickers as columns, and daily Returns as values.

| Date       | AAPL   | MSFT   | GOOGL  | AMZN   | JPM    |
|------------|--------|--------|--------|--------|--------|
| 2024-01-02 | 0.0123 | 0.0087 | -0.0041| 0.0156 | -0.0023|
| 2024-01-03 | -0.0045| 0.0021 | 0.0098 | -0.0067| 0.0034 |

Step 2: Calculate Key Metrics

- Pairwise correlation matrix between all tickers using 30-day rolling windows
- Annualized volatility for each ticker: std(daily_returns) × √252
- Sharpe Ratio for each ticker: (annualized_returns - 0.03) / annualized_volatility

**Pairwise Correlation Matrix** — using 30-day rolling windows:

$$
\rho_{XY} = \frac{\text{Cov}(R_X, R_Y)}{\sigma_X \sigma_Y}
$$

**Annualized Volatility:**

$$
\sigma_{\text{ann}} = \sigma_{\text{daily}} \times \sqrt{252}
$$

**Sharpe Ratio** (assuming risk-free rate = 3%):

$$
\text{Sharpe} = \frac{R_{\text{ann}} - 0.03}{\sigma_{\text{ann}}}
$$


| Ticker | Volatility | Return | Sharpe |
|--------|------------|--------|--------|
| AAPL   | 22.3%      | 15.6%  | 0.56   |
| MSFT   | 20.1%      | 18.2%  | 0.76   |
| GOOGL  | 19.8%      | 14.1%  | 0.56   |
| AMZN   | 25.4%      | 22.3%  | 0.74   |
| JPM    | 18.7%      | 12.8%  | 0.52   |

Step 3: Construct Minimum Variance Portfolio

For each date:

- Select the **top 5 tickers** with the highest Sharpe Ratio on that date
- Assign weights **inversely proportional to volatility**:

$$
w_i = \frac{1 / \sigma_i}{\sum_{j=1}^{5} 1 / \sigma_j}
$$

- Calculate portfolio daily return:

$$
R_p = \sum_{i=1}^{5} w_i \cdot R_i
$$

- Compute cumulative performance over time.

| Metric              | Min Var Port | Equal Weight |
|---------------------|--------------|--------------|
| Annualized Return   | 16.8%        | 15.2%        |
| Annualized Vol      | 12.4%        | 18.7%        |
| Sharpe Ratio        | 1.11         | 0.65         |
| Cumulative Return   | 52.3%        | 44.1%        |


Step 4: Identify High Correlation Periods

Identify periods where **correlation between JPM and SPY > 0.8** and analyze market conditions (e.g., VIX level, volatility, market stress).

| Period     | Correlation | Market Cond. | VIX Level |
|------------|-------------|--------------|-----------|
| 2024-03-10 | 0.87        | High Vol     | 25.3      |
| 2024-04-15 | 0.92        | Market Stress| 32.1      |


In [10]:
# step 1
pivot_data = data.pivot_table(index=['date'], columns=['ticker'], values='returns', aggfunc='mean')

In [14]:
# step 2

# rolling correlation
tickers = pivot_data.columns.tolist()
pairs = list(combinations(tickers, 2))

corr_list = []
for ticker1, ticker2 in pairs:
    rolling_corr = pivot_data[ticker1].rolling(window=30).corr(pivot_data[ticker2])

    for index, value in rolling_corr.dropna().items():
        corr_list.append((ticker1, ticker2, index, value))
            
corr_df = pd.DataFrame(corr_list, columns=['ticker1', 'ticker2', 'date', 'correlation'])

# annualized volatility ==> std(daily_returns) × √252
data_metrics = data.groupby(['ticker']).agg(
    volatility = ('returns', 'std'),
    returns = ('returns', 'mean')    
)
data_metrics['volatility'] = data_metrics['volatility'] * np.sqrt(252)
data_metrics['returns'] = data_metrics['returns'] * np.sqrt(252)


# sharpe ratio
data_metrics['sharpe_ratio'] = (data_metrics['returns'] - 0.03 ) / data_metrics['volatility']